# 🎯 [ISR03] 드론 비행로그 패턴 이상 탐지와 운용 주체 단정 방지
### Google Colab 실습 환경 — 미션 03 (isr03_m03_student.ipynb) (정보·감시·정찰(ISR) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR03/ISR03_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 상용 드론이 정찰과 공격에 폭넓게 쓰이면서, 노획하거나 수집한 드론 비행로그를 분석해 이상 비행을 찾아내는 일이 중요해졌다. 그러나 같은 상용 기체를 아군·민간·적대 세력이 모두 쓰기 때문에, 비행 특성만 보고 운용 주체를 성급하게 단정하면 오판으로 이어질 수 있다. 본 프로젝트에서는 교육용 가상 비행로그를 코드로 직접 만들어 스키마 정의, GPS 스푸핑 보정 전처리, 규칙 기반 Baseline, 군집·비지도 학습 기반 이상 탐지, 불확실성을 반영한 운용 주체 신뢰도 엔진, 분석 대시보드, 정보분석관 심층 검토(HITL), 대량 로그 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 이상 비행을 놓치지 않으면서 오탐을 줄이고, 운용 주체에 관한 판단은 항상 확률·불확실성·근거와 함께 사람의 검토를 거쳐 보고되도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 🛠️ 2. 실습 실행 및 산출물 저장·데이터 연계 가이드

1. **Colab / 로컬 실행 환경 안내**
   - 아래 환경 설정 셀의 `!pip install -q ...` 구문은 **Google Colab / Jupyter 노트북 전용** 매직 커맨드입니다.
   - 일반 로컬 Python(`.py`) 환경에서 실행할 때는 해당 줄을 제거하고 터미널에서 `pip install numpy pandas matplotlib scipy scikit-learn`을 먼저 실행하세요.

2. **산출물(`deliverables`) 디렉토리 구조 및 파일 저장 방법**
   - 본 노트북의 각 미션 셀은 Colab 환경에서 단독 실행이 가능하도록 인메모리 시뮬레이션으로 구성되어 있습니다.
   - 프로젝트 제출용 산출물(`data/schema/`, `docs/`, `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `tests/`, `configs/`)을 실제 파일로 저장하려면, 환경 설정 셀에 제공된 `init_deliverable_dirs()` 및 `save_deliverable(rel_path, content)` 헬퍼 함수 또는 셀 상단의 `%%writefile <파일경로>` 매직 커맨드를 사용하세요.

3. **첨부 데이터셋(`isr03_mXX_dataset.json`) 연계 활용**
   - 각 미션 셀은 외부 파일 의존성 없이도 독립 실행되도록 자체 난수 기반 가상 데이터를 생성합니다.
   - 미션별 첨부파일(`isr03_m01_dataset.json` ~ `isr03_m10_dataset.json`, `isr03_mXX_sample.md`)은 `load_mission_dataset(1)` 함수로 불러와 스키마 대조, 경계값 추가 검증, 파라미터 확장 실험에 연계 활용할 수 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# 산출물 디렉토리 초기화 및 파일 저장/첨부 데이터셋 로드 헬퍼
DELIVERABLE_DIRS = [
    'data/schema', 'docs', 'src/preprocess', 'src/models',
    'src/verification', 'src/ui', 'src/hitl', 'tests', 'configs'
]

def init_deliverable_dirs(base_dir='.'):
    """프로젝트 필수 산출물 디렉토리를 생성한다."""
    for d in DELIVERABLE_DIRS:
        os.makedirs(os.path.join(base_dir, d), exist_ok=True)

def save_deliverable(rel_path, content, base_dir='.'):
    """산출물 파일(JSON, YAML, MD, PY 등)을 지정 경로에 저장한다."""
    full_path = os.path.join(base_dir, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(content if isinstance(content, str) else json.dumps(content, ensure_ascii=False, indent=2))
    return full_path

def load_mission_dataset(mission_num, base_dir='.'):
    """첨부된 미션별 가상 데이터셋(isr03_mXX_dataset.json)을 불러온다."""
    fname = f'isr03_m{int(mission_num):02d}_dataset.json'
    for candidate in [os.path.join(base_dir, fname), os.path.join(base_dir, 'uploads', 'ISR03', fname)]:
        if os.path.exists(candidate):
            with open(candidate, encoding='utf-8') as f:
                return json.load(f)
    raise FileNotFoundError(f'{fname} 파일을 찾을 수 없습니다. Colab에 업로드했는지 확인하세요.')


---

## 🎯 미션 03. 드론 통상 비행 패턴 정의 및 규칙 기반 이상 비행(급선회·호버링) 탐지 Baseline 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

통상적인 정찰 비행 패턴을 정의하고 임의 경로 이탈, 급격한 고도 하강 등 이상 궤적을 1차 식별하는 Baseline 규칙 기반 모델을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **통상 정찰 비행 패턴을 수치 기준으로 정의하고 급선회·호버링·급강하·경로 이탈·비인가 구역 진입을 1차 식별하는 규칙 기반 Baseline을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **선회율, 저속 정지 지속시간, 5초간 고도 하강량, 계획 경로 대비 이탈 거리, 비인가 구역 진입 여부를 규칙으로 만들어 기록 단위로 판정하고 비행 단위로 모은다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 5개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/models/isr03_baseline_model.py 및 기본 추론 결과 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 알고리즘의 논리적 흐름 및 군사적 판단 기준의 투명성

#### 미션 PASS 조건

- 정량 기준: 기본 데이터셋 대상 탐지/처리 정확도 75% 이상 달성


- 진행 기준: 단위 기능 동작 확인 및 기본 시나리오 실행 로그 생성


- 검증 방법: 기준 벤치마크 데이터셋 대상 추론 결과 오차율 비교


- AI 검증 기준: AI의 오탐 개선·임계값 제안을 벤치마크 실험표의 정확도·미탐률로 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""통상 정찰 비행 패턴(순찰·측량 경로, 일정 고도·속도)을 기준으로 급선회·호버링·급강하·경로 이탈·비인가 구역 진입을 1차 식별하는 규칙 기반 이상 비행 탐지 Baseline을 구현하고, 비행 단위 정확도와 판단 지연시간을 측정한다."""
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 합성 드론 비행로그 생성기 (실제 좌표·기체·부대 정보 없음)
# 가상 구역(원점 기준 동·북 방향 m 좌표)에서 1초 간격 비행로그와 정답(이상 구간·운용 주체)을 만든다.
# ---------------------------------------------------------------------------
NFZ = {"x": 2500.0, "y": 2500.0, "r": 700.0}        # 가상 비인가 비행구역
WATCH = {"x": -2000.0, "y": 1500.0, "r": 600.0}     # 가상 관심 구역
SCENARIOS = ["sharp_turn", "hover", "dive", "deviation", "nfz_entry",
             "low_level", "rc_silent", "infiltration", "loiter", "speed_burst"]
OPERATORS = ["friendly", "civil", "hostile"]
# 시나리오별 운용 주체 분포: 일부러 크게 겹치게 설계했다(비행 특성만으로 운용 주체를 단정할 수 없다).
OPERATOR_P = {"normal": (.45, .47, .08), "survey": (.15, .82, .03), "sharp_turn": (.40, .35, .25),
              "hover": (.25, .45, .30), "dive": (.25, .25, .50), "deviation": (.30, .40, .30),
              "nfz_entry": (.10, .50, .40), "low_level": (.35, .15, .50), "rc_silent": (.30, .15, .55),
              "infiltration": (.12, .08, .80), "loiter": (.30, .20, .50), "speed_burst": (.30, .20, .50)}
PLATFORM_P = {"friendly": (.40, .55, .05), "civil": (.88, .02, .10), "hostile": (.55, .30, .15)}   # commercial/military/unknown
FIRMWARE_P = {"friendly": (.30, .65, .05), "civil": (.85, .10, .05), "hostile": (.20, .50, .30)}   # stock/custom/unknown


def _wrap(a):
    # [TODO : 학생 실습 3] 드론 통상 비행 패턴 정의 및 규칙 기반 이상 비행(급선회·호버링) 탐지 Baseline 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _bearing(p, q):
    # [TODO : 학생 실습 3] 드론 통상 비행 패턴 정의 및 규칙 기반 이상 비행(급선회·호버링) 탐지 Baseline 구현 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _away_from_nfz(rng, min_d):
    while True:
        c = rng.uniform(-3800, 3800, 2)
        if np.hypot(c[0] - NFZ["x"], c[1] - NFZ["y"]) > min_d:
            return c


def make_flight(fid, kind, rng, n=300, weather=None):
    """비행 1회 = (로그 DataFrame, 메타 dict). kind: normal / survey / SCENARIOS 중 하나"""
    weather = weather or str(rng.choice(["clear", "night", "windy", "rain"], p=[.4, .25, .2, .15]))
    wind = {"clear": 1.0, "night": 1.0, "windy": 2.5, "rain": 1.8}[weather]
    v0, alt0 = rng.uniform(11, 17), rng.uniform(70, 150)
    nfz_c = np.array([NFZ["x"], NFZ["y"]])
    if kind == "infiltration":                      # 원거리에서 비인가 구역 방향으로 저고도 직진
        ang = rng.uniform(0, 2 * np.pi)
        start = nfz_c + 4000 * np.array([np.cos(ang), np.sin(ang)])
        plan = np.array([start, nfz_c])
        v0, alt0 = rng.uniform(9, 11), rng.uniform(25, 35)
    else:
        if kind == "nfz_entry":                     # 비인가 구역 1.6km 밖에서 순찰하다가 진입
            ang = rng.uniform(0, 2 * np.pi)
            c = nfz_c + 1600 * np.array([np.cos(ang), np.sin(ang)])
        else:
            c = _away_from_nfz(rng, 2600)
        if kind == "survey":                        # 민간 측량형 왕복(지그재그) 경로
            w, h = rng.uniform(500, 700), 120.0
            plan = np.array([c + [(-w if i % 4 in (0, 3) else w), i // 2 * h - 360] for i in range(8)])
        else:                                       # 사각 순찰 경로(웨이포인트 4개)
            a, b, th = rng.uniform(300, 600), rng.uniform(200, 400), rng.uniform(0, np.pi)
            R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
            plan = c + np.array([[-a, -b], [a, -b], [a, b], [-a, b]]) @ R.T
    s0 = int(rng.integers(80, 160))
    span = {"sharp_turn": 30, "hover": 60, "dive": 8, "deviation": 70, "nfz_entry": 120, "low_level": 120,
            "rc_silent": 120, "infiltration": n - 20, "loiter": 90, "speed_burst": 25}.get(kind, 0)
    if kind == "infiltration":
        s0 = 20
    win = (s0, s0 + span)
    turns = set(rng.choice(np.arange(s0, s0 + 30), 3, replace=False).tolist()) if kind == "sharp_turn" else set()
    # 난수를 미리 한 번에 뽑아 두고(속도 향상) 1초 단위로 기체 운동을 적분한다.
    e_h, e_v, e_a = rng.normal(0, 0.8 * wind, n), rng.normal(0, 0.3 * wind, n), rng.normal(0, 0.4 * wind, n)
    rc_arr = np.clip(rng.normal(rng.uniform(60, 92), 6, n), 0, 100)
    gust = (rng.random(n) < (0.004 if weather == "windy" else 0.0)) * rng.uniform(20, 35, n)
    burst, turn_amt = rng.uniform(23, 28), rng.choice([-1, 1], n) * rng.uniform(100, 160, n)
    side = float(rng.choice([-90, 90]))
    low_alt, rng_rc = rng.uniform(35, 55), rng.uniform(0, 20, n)          # 저고도 목표, 조종신호 두절 시 잔여 수신값
    x, y, wi = float(plan[0][0]), float(plan[0][1]), 1
    hdg, alt = _bearing((x, y), plan[1]), alt0
    max_rate = 40.0 if kind == "survey" else 12.0
    dev_target, X, Y, A, V, HD = None, np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n)
    for t in range(n):
        inw = win[0] <= t < win[1]
        tx, ty = plan[1] if kind == "infiltration" else plan[wi % len(plan)]
        spd_cmd, alt_cmd = v0, alt0
        if inw:
            if kind == "deviation":
                if dev_target is None:
                    r = math.radians(hdg + side)
                    dev_target = (x + 900 * math.sin(r), y + 900 * math.cos(r))
                tx, ty = dev_target
            elif kind == "nfz_entry":
                tx, ty = NFZ["x"], NFZ["y"]
            elif kind == "hover":
                spd_cmd = 0.0
            elif kind == "speed_burst":
                spd_cmd = burst
            elif kind == "low_level":
                alt_cmd = low_alt
            if kind in ("rc_silent", "infiltration"):
                rc_arr[t] = rng_rc[t]
        if inw and kind == "loiter":
            hdg = (hdg + 9.0 + e_h[t] / wind) % 360
        else:
            d = _wrap(math.degrees(math.atan2(tx - x, ty - y)) - hdg)
            hdg = (hdg + max(-max_rate, min(max_rate, d)) + e_h[t]) % 360
        if t in turns:
            hdg = (hdg + turn_amt[t]) % 360
        v = abs(spd_cmd + e_v[t]) if spd_cmd > 0 else abs(e_v[t] / wind)
        x += v * math.sin(math.radians(hdg)); y += v * math.cos(math.radians(hdg))
        if kind != "infiltration" and math.hypot(x - tx, y - ty) < 40 and not (inw and kind in ("deviation", "nfz_entry")):
            wi += 1
        if inw and kind == "dive":
            alt -= 9.0 + e_v[t]
        else:
            alt += max(-3.0, min(3.0, alt_cmd - alt)) + e_a[t]
        alt -= gust[t]                                     # 돌풍 하강기류(정상 비행에서도 발생)
        if kind == "dive" and t >= win[1]:
            alt0 = min(alt0, alt)
        alt = max(alt, 3.0)
        X[t], Y[t], A[t], V[t], HD[t] = x, y, alt, v, hdg
    gps_sd = {"clear": 2.0, "night": 2.0, "windy": 3.0, "rain": 4.5}[weather]     # 악천후일수록 GPS 잡음(다중경로) 증가
    df = pd.DataFrame({"t": np.arange(n), "x": X + rng.normal(0, gps_sd, n), "y": Y + rng.normal(0, gps_sd, n),
                       "alt": A + rng.normal(0, 0.5, n), "speed": V + rng.normal(0, 0.2, n),
                       "heading": (HD + rng.normal(0, 1.5, n)) % 360, "rc": rc_arr.round(1),
                       "sats": rng.integers(10, 15, n), "hdop": rng.uniform(0.6, 1.2, n).round(2),
                       "x_true": X, "y_true": Y})
    df["anomaly"] = (df["t"] >= win[0]) & (df["t"] < win[1]) & (kind in SCENARIOS)
    op = OPERATORS[int(rng.choice(3, p=OPERATOR_P[kind]))]
    meta = {"id": fid, "kind": kind, "weather": weather, "operator": op, "plan": plan, "window": win if kind in SCENARIOS else None,
            "platform": ["commercial", "military", "unknown"][int(rng.choice(3, p=PLATFORM_P[op]))],
            "firmware": ["stock", "custom", "unknown"][int(rng.choice(3, p=FIRMWARE_P[op]))]}
    return df, meta


def corrupt(df, rng, spoof=True, emi=True, drop=True, dup=True):
    """전장 잡음 주입: GPS 스푸핑(위치 점프·표류), 전자기 간섭(고도 스파이크), 센서 단절(결측), 중복 기록."""
    d = df.copy()
    d["spoof_true"] = False
    if spoof:
        a = int(rng.integers(40, len(d) - 60)); L = int(rng.integers(25, 45))
        th = rng.uniform(0, 2 * np.pi)
        drift = rng.uniform(22, 28) * np.arange(L)[:, None] * np.array([np.cos(th), np.sin(th)])
        jump = rng.uniform(250, 400) * np.array([np.cos(th + 1), np.sin(th + 1)])
        d.loc[a:a + L - 1, ["x", "y"]] += drift + jump
        d.loc[a:a + L - 1, "spoof_true"] = True
    if emi:
        idx = rng.choice(len(d), 6, replace=False)
        d.loc[idx, "alt"] += rng.choice([-1, 1], 6) * rng.uniform(60, 150, 6)
    if drop:
        a = int(rng.integers(10, len(d) - 20))
        d.loc[a:a + int(rng.integers(4, 9)), ["x", "y", "alt", "speed", "heading"]] = np.nan
        d = d.drop(index=rng.choice(len(d), 5, replace=False))
    if dup:
        d = pd.concat([d, d.sample(5, random_state=int(rng.integers(1e6)))]).sort_values("t", kind="stable")
    return d.reset_index(drop=True)


def make_dataset(n=200, seed=7, anomaly_ratio=0.5, scenarios=SCENARIOS):
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n):
        if i % 10 < anomaly_ratio * 10:
            kind = scenarios[(i // 10 * int(anomaly_ratio * 10) + i % 10) % len(scenarios)]
        else:
            kind = "survey" if i % 10 == 9 else "normal"
        df, meta = make_flight(f"FL{i:04d}", kind, rng)
        out.append((df, meta))
    return out
# ---------------------------------------------------------------------------

# ---- 전처리·규칙 탐지 공통 함수 ----
SPOOF_THR = {"clear": 12.0, "night": 12.0, "windy": 15.0, "rain": 19.0}   # 기상별 GPS-추측항법 불일치 허용치(m)


def clean(df, weather="clear"):
    """전처리: 중복 제거·정렬 -> 1초 재표본화 -> 결측 보간 -> 고도 스파이크 제거 -> GPS 스푸핑 구간 추측항법 보정."""
    d = df.drop_duplicates("t").sort_values("t").set_index("t")
    d = d.reindex(np.arange(int(d.index.min()), int(d.index.max()) + 1))
    n_missing = int(d["x"].isna().sum())
    num = ["x", "y", "alt", "speed", "rc", "sats", "hdop"]
    d[num] = d[num].interpolate(limit_direction="both")
    hd = d["heading"].dropna()                                   # 방위는 0/360 경계를 풀어서 보간
    un = pd.Series(np.degrees(np.unwrap(np.radians(hd.to_numpy(float)))), index=hd.index)
    d["heading"] = un.reindex(d.index).interpolate(limit_direction="both").to_numpy() % 360
    med = d["alt"].rolling(5, center=True, min_periods=1).median()
    spike = (d["alt"] - med).abs() > 25
    d.loc[spike, "alt"] = med[spike]
    x, y = d["x"].to_numpy(float), d["y"].to_numpy(float)
    r = np.radians(d["heading"].to_numpy(float)); v = d["speed"].to_numpy(float)
    dr = np.c_[v * np.sin(r), v * np.cos(r)]                  # 보고 속도·방위로 계산한 1초 이동량(추측항법)
    xc, yc, flag = x.copy(), y.copy(), np.zeros(len(x), bool)
    thr = SPOOF_THR.get(weather, 12.0)
    ok_run = 0
    for i in range(1, len(x)):
        px, py = xc[i - 1] + dr[i, 0], yc[i - 1] + dr[i, 1]          # 추측항법 예상 위치
        step_ok = np.hypot(x[i] - x[i - 1] - dr[i, 0], y[i] - y[i - 1] - dr[i, 1]) <= thr
        ok_run = ok_run + 1 if step_ok else 0
        if flag[i - 1]:
            # 스푸핑 구간: GPS가 예상 위치로 돌아오거나, GPS 이동량이 5초 연속 물리적으로 맞으면 GPS로 복귀
            if np.hypot(x[i] - px, y[i] - py) < thr * 2 or ok_run >= 5:
                xc[i], yc[i] = x[i], y[i]
            else:
                xc[i], yc[i], flag[i] = px, py, True
        elif not step_ok:
            xc[i], yc[i], flag[i] = px, py, True                     # GPS 이동량이 물리적으로 맞지 않음 -> 스푸핑 의심
        else:
            xc[i], yc[i] = xc[i - 1] + x[i] - x[i - 1], yc[i - 1] + y[i] - y[i - 1]
    d["x"], d["y"], d["spoof_flag"] = xc, yc, flag
    d = d.reset_index()
    d.attrs.update({"missing": n_missing, "spikes": int(spike.sum()), "spoof_pts": int(flag.sum())})
    return d


def xtrack(px, py, plan):
    """계획 경로(웨이포인트 폐곡선) 대비 이탈 거리(m)."""
    P = np.asarray(plan, float)
    segs = [(P[i], P[(i + 1) % len(P)]) for i in range(len(P) if len(P) > 2 else 1)]
    best = np.full(len(px), np.inf)
    for a, b in segs:
        ab = b - a
        u = np.clip(((px - a[0]) * ab[0] + (py - a[1]) * ab[1]) / max(ab @ ab, 1e-9), 0, 1)
        best = np.minimum(best, np.hypot(px - a[0] - u * ab[0], py - a[1] - u * ab[1]))
    return best


RULES = {"turn_rate": 45.0, "hover_speed": 1.5, "hover_sec": 20, "dive_m": 30.0, "xtrack_m": 250.0}


def rule_flags(d, plan, R=RULES):
    """규칙 기반 이상 비행 판정(기록 단위). 급선회·호버링·급강하·경로 이탈·비인가 구역 진입."""
    hr = np.abs(_wrap(np.diff(d["heading"].to_numpy(float), prepend=d["heading"].iloc[0])))
    slow = (d["speed"] < R["hover_speed"]).to_numpy()
    run = np.zeros(len(slow), int)
    for i in range(len(slow)):
        run[i] = run[i - 1] + 1 if slow[i] and i else int(slow[i])
    hover = np.zeros(len(slow), bool)
    for i in np.where(run >= R["hover_sec"])[0]:
        hover[i - R["hover_sec"] + 1:i + 1] = True
    alt = d["alt"].to_numpy(float)
    drop = alt - np.r_[alt[5:], np.full(5, alt[-1])]
    return {"급선회": hr > R["turn_rate"], "호버링": hover, "급강하": drop > R["dive_m"],
            "경로이탈": xtrack(d["x"].to_numpy(float), d["y"].to_numpy(float), plan) > R["xtrack_m"],
            "비인가구역": np.hypot(d["x"] - NFZ["x"], d["y"] - NFZ["y"]).to_numpy() < NFZ["r"]}


def run_baseline(data, R=RULES):
    out = []
    for df, m in data:
        t0 = time.perf_counter()
        d = clean(df, m["weather"])
        f = rule_flags(d, m["plan"], R)
        hits = [k for k, v in f.items() if v.any()]
        out.append({"id": m["id"], "kind": m["kind"], "truth": m["kind"] in SCENARIOS, "pred": bool(hits),
                    "rules": ",".join(hits) or "-", "ms": (time.perf_counter() - t0) * 1000})
    return pd.DataFrame(out)


# 통상 비행 패턴 정의(판단 기준을 숫자로 공개해 투명성 확보)
print("[통상 비행 기준] 선회율 45도/초 이하, 1.5m/s 미만 정지 20초 미만, 5초간 고도 하강 30m 이하, 계획 경로 250m 이내, 비인가 구역 밖")

# 1) 기본 데이터셋: 정상(순찰·측량) 50% + 기본 이상 5종 50%
basic = make_dataset(100, seed=11, scenarios=SCENARIOS[:5])
r = run_baseline(basic)
acc = (r["truth"] == r["pred"]).mean()
tp, fp = int((r.truth & r.pred).sum()), int((~r.truth & r.pred).sum())
fn, tn = int((r.truth & ~r.pred).sum()), int((~r.truth & ~r.pred).sum())
print(f"\n기본 데이터셋 100건: 정확도 {acc:.1%} (목표 75% 이상: {'달성' if acc >= 0.75 else '미달'}), TP {tp} / FP {fp} / FN {fn} / TN {tn}")
print(f"비행 1건 판단 지연시간 평균 {r['ms'].mean():.1f} ms")

print("\n[유형별 탐지율과 주로 걸린 규칙]")
g = r.groupby("kind").agg(건수=("id", "size"), 이상판정률=("pred", "mean"), 대표규칙=("rules", lambda s: s.value_counts().index[0]))
print(g.round(2).to_string())

# 2) 급선회 임계값 민감도(휴리스틱 가중치 1차 조정)
print("\n선회율 임계값  정확도  정상 비행 오탐")
for thr in [30, 45, 60, 90]:
    rr = run_baseline(basic, {**RULES, "turn_rate": thr})
    print(f"{thr:>8}도/초   {(rr.truth == rr.pred).mean():.1%}   {int((~rr.truth & rr.pred).sum())}건")

# 3) 한계: 은밀·복합 이상 5종에는 규칙이 거의 반응하지 않는다(미션 4의 동기)
hard = make_dataset(60, seed=12, scenarios=SCENARIOS[5:])
h = run_baseline(hard)
print("\n[복합 이상 5종 탐지율]", h[h.truth].groupby("kind")["pred"].mean().round(2).to_dict())

# 4) 기본 시나리오 실행 로그 예시와 그림
ex = next((df, m) for df, m in basic if m["kind"] == "sharp_turn")
d = clean(ex[0], ex[1]["weather"]); f = rule_flags(d, ex[1]["plan"])
print(f"\n실행 로그 예시 {ex[1]['id']}: 급선회 기록 {int(f['급선회'].sum())}건 (t={list(d['t'][f['급선회']][:3])}) -> '이상 비행 의심, 분석관 확인 필요'")
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, (df, m) in zip(ax, [ex, next((df, m) for df, m in basic if m["kind"] == "deviation")]):
    d = clean(df, m["weather"]); fl = np.any(list(rule_flags(d, m["plan"]).values()), axis=0)
    P = np.vstack([m["plan"], m["plan"][:1]])
    a.plot(P[:, 0], P[:, 1], "k--", lw=1, label="planned route")
    a.plot(d["x"], d["y"], "-", color="tab:blue", lw=1, label="flight log")
    a.plot(d["x"][fl], d["y"][fl], "o", color="tab:red", ms=4, label="rule hit")
    a.set_title(f"{m['id']} ({m['kind']})"); a.axis("equal"); a.legend(fontsize=8); a.set_xlabel("East (m)"); a.set_ylabel("North (m)")
plt.suptitle("Rule-based baseline detections (synthetic logs)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[통상 비행 기준] 선회율 45도/초 이하, 1.5m/s 미만 정지 20초 미만, 5초간 고도 하강 30m 이하, 계획 경로 250m 이내, 비인가 구역 밖

기본 데이터셋 100건: 정확도 91.0% (목표 75% 이상: 달성), TP 49 / FP 8 / FN 1 / TN 42
비행 1건 판단 지연시간 평균 10.0 ms

[유형별 탐지율과 주로 걸린 규칙]
            건수  이상판정률        대표규칙
kind                             
deviation   10   1.00        경로이탈
dive        10   1.00         급강하
hover       10   1.00         호버링
nfz_entry   10   0.90  경로이탈,비인가구역
normal      40   0.12           -
sharp_turn  10   1.00         급선회
survey      10   0.30           -

선회율 임계값  정확도  정상 비행 오탐
      30도/초   84.0%   15건
      45도/초   91.0%   8건
      60도/초   93.0%   6건
      90도/초   93.0%   6건

[복합 이상 5종 탐지율] {'infiltration': 0.0, 'loiter': 0.17, 'low_level': 0.17, 'rc_silent': 0.17, 'speed_burst': 0.0}

실행 로그 예시 FL0000: 급선회 기록 3건 (t=[158, 166, 168]) -> '이상 비행 의심, 분석관 확인 필요'
```
</details>
